# Assignment 3 · Parte 1 · Visualización

**¿Dónde llovió más en el primer semestre de 2022, y coincide eso con dónde
se declaró emergencia?**

Grupo 9 · Diplomado PUCP 2026

## 1. Cargar los datos

Usamos la `tabla_final.csv` del Assignment 2 (opción B), guardada aquí como
`datos/dataset.csv`. Una fila por departamento.

El `ubigeo` se lee como **texto** (`dtype=str`): si se lee como número,
`01` se convierte en `1` y el cruce con el mapa falla en la Parte 2.

In [1]:
import pandas as pd
import plotly.express as px

In [2]:
datos = pd.read_csv("datos/dataset.csv", dtype={"ubigeo": str})

In [3]:
print("Filas y columnas:", datos.shape)
datos.head()

Filas y columnas: (25, 9)


,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,436.5,1,1,0
1,02,Áncash,Huaraz,-9.52614,-77.52869,975.3,7,0,0
2,03,Apurímac,Abancay,-13.63426,-72.88380,711.8,2,0,0
3,04,Arequipa,Arequipa,-16.39899,-71.53747,189.9,0,0,0
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,485.1,0,1,0


## 2. Tema y columnas

**Pregunta:** ¿los departamentos donde más llovió entre enero y mayo de 2022
son los mismos donde la PCM declaró estado de emergencia?

| Columna | Qué mide | Fuente |
|---|---|---|
| `lluvia_total_mm` | lluvia acumulada ene-may 2022 en la capital del departamento | Open-Meteo (reanálisis) |
| `dias_lluvia_fuerte` | días con lluvia fuerte en ese periodo | Open-Meteo (reanálisis) |
| `declaratorias` | decretos que **declaran** emergencia por lluvias | PCM, gob.pe |
| `prorrogas` | decretos que **prorrogan** una emergencia | PCM, gob.pe |

## 3. Gráfico de barras: los 10 departamentos con más lluvia

Antes de graficar, ordenamos y nos quedamos con los 10 primeros. Así el
gráfico y la tabla dicen lo mismo, y podemos comprobarlo en el paso 6.

In [4]:
top_lluvia = datos.sort_values("lluvia_total_mm", ascending=False).head(10)
top_lluvia[["departamento", "lluvia_total_mm"]]

,departamento,lluvia_total_mm
15,Loreto,1581.8
24,Ucayali,1288.8
1,Áncash,975.3
11,Junín,910.4
16,Madre de Dios,908.0
8,Huancavelica,803.8
21,San Martín,789.8
20,Puno,779.5
18,Pasco,774.0
5,Cajamarca,773.3


In [5]:
fuente = "Fuente: Open-Meteo (reanálisis) y PCM en gob.pe, enero-mayo 2022"

grafico_barras = px.bar(
    top_lluvia,
    x="lluvia_total_mm",
    y="departamento",
    orientation="h",
    title="Loreto, Ucayali y Áncash acumularon la mayor lluvia entre enero y mayo de 2022",
    labels={"lluvia_total_mm": "Lluvia acumulada (mm)", "departamento": "Departamento"},
)

# Plotly dibuja la primera barra abajo; invertimos para que Loreto quede arriba.
grafico_barras.update_yaxes(autorange="reversed")
grafico_barras.add_annotation(
    text=fuente, xref="paper", yref="paper", x=0, y=-0.2, showarrow=False
)
grafico_barras.update_layout(template="plotly_white")
grafico_barras.show()

## 4. Gráfico de dispersión: lluvia total contra días de lluvia fuerte

Cada punto es un departamento. `hover_name` permite leer su nombre al pasar
el cursor. Coloreamos según si tuvo o no una declaratoria de emergencia.

In [6]:
# Una columna de texto es más legible en la leyenda que un 0 o un 1.
datos["emergencia"] = "Sin declaratoria"
datos.loc[datos["declaratorias"] > 0, "emergencia"] = "Con declaratoria"
datos["emergencia"].value_counts()

emergencia
Sin declaratoria    22
Con declaratoria     3
Name: count, dtype: int64

In [7]:
grafico_dispersion = px.scatter(
    datos,
    x="lluvia_total_mm",
    y="dias_lluvia_fuerte",
    color="emergencia",
    hover_name="departamento",
    title="Donde llovió más hubo más días de lluvia fuerte, pero las emergencias no están ahí",
    labels={
        "lluvia_total_mm": "Lluvia acumulada (mm)",
        "dias_lluvia_fuerte": "Días de lluvia fuerte",
        "emergencia": "",
    },
)
grafico_dispersion.add_annotation(
    text=fuente, xref="paper", yref="paper", x=0, y=-0.2, showarrow=False
)
grafico_dispersion.update_layout(template="plotly_white")
grafico_dispersion.show()

## 5. Tercer gráfico: lluvia con y sin declaratoria

Un boxplot compara los dos grupos. Con solo 3 departamentos con
declaratoria, `points="all"` dibuja cada punto para no esconder que son pocos.

In [8]:
grafico_cajas = px.box(
    datos,
    x="emergencia",
    y="lluvia_total_mm",
    points="all",
    hover_name="departamento",
    title="Los 3 departamentos con declaratoria no están entre los más lluviosos",
    labels={"emergencia": "", "lluvia_total_mm": "Lluvia acumulada (mm)"},
)
grafico_cajas.add_annotation(
    text=fuente, xref="paper", yref="paper", x=0, y=-0.2, showarrow=False
)
grafico_cajas.update_layout(template="plotly_white")
grafico_cajas.show()

## 6. Verificación: los extremos de la tabla coinciden con los gráficos

Un gráfico bonito puede estar mal. Miramos los valores más altos y más bajos
en la tabla y los comparamos con lo que se ve arriba.

In [9]:
print("Los 3 con más lluvia")
datos.nlargest(3, "lluvia_total_mm")[["departamento", "lluvia_total_mm", "dias_lluvia_fuerte"]]

Los 3 con más lluvia


,departamento,lluvia_total_mm,dias_lluvia_fuerte
15,Loreto,1581.8,25
24,Ucayali,1288.8,20
1,Áncash,975.3,7


In [10]:
print("Los 3 con menos lluvia")
datos.nsmallest(3, "lluvia_total_mm")[["departamento", "lluvia_total_mm", "dias_lluvia_fuerte"]]

Los 3 con menos lluvia


,departamento,lluvia_total_mm,dias_lluvia_fuerte
22,Tacna,11.5,0
17,Moquegua,17.8,0
10,Ica,25.1,0


In [11]:
print("Departamentos con declaratoria")
datos[datos["declaratorias"] > 0][["departamento", "lluvia_total_mm", "declaratorias"]]

Departamentos con declaratoria


,departamento,lluvia_total_mm,declaratorias
0,Amazonas,436.5,1
4,Ayacucho,485.1,1
19,Piura,89.2,1


**Qué comprobamos:**

- Loreto (1,581.8 mm), Ucayali (1,288.8 mm) y Áncash (975.3 mm) son los tres
  primeros de la tabla y las tres barras más largas del gráfico.
- Tacna (11.5 mm) y Moquegua (17.8 mm) son los más secos; en el gráfico de
  dispersión están pegados al origen.
- **Lima y Callao tienen exactamente 39.2 mm.** No es un error de copia: sus
  capitales caen en la misma celda del modelo de reanálisis. Hay que
  recordarlo al interpretar el mapa.
- Los tres con declaratoria son Amazonas, Ayacucho y Piura. Ninguno está
  entre los 3 más lluviosos.

## 7. Exportar un gráfico a HTML

El HTML conserva la interactividad (zoom, hover) y se puede enviar por correo
o abrir en cualquier navegador.

In [12]:
# "cdn" evita guardar plotly.js dentro del archivo (4 MB menos).
grafico_barras.write_html("grafico_top_lluvia.html", include_plotlyjs="cdn")
print("Guardado: grafico_top_lluvia.html")

Guardado: grafico_top_lluvia.html


## Qué hicimos

| Paso | Qué hicimos | Código |
|---|---|---|
| 1 | Leer el CSV con ubigeo como texto | `pd.read_csv(..., dtype={"ubigeo": str})` |
| 2 | Ordenar y quedarnos con el top 10 | `sort_values(...).head(10)` |
| 3 | Barras horizontales | `px.bar(..., orientation="h")` |
| 4 | Dispersión con nombre al pasar el cursor | `px.scatter(..., hover_name=...)` |
| 5 | Comparar dos grupos | `px.box(..., points="all")` |
| 6 | Verificar extremos contra la tabla | `nlargest`, `nsmallest` |
| 7 | Exportar a HTML | `fig.write_html(...)` |